# 04 · SQL Analysis (demonstration notebook)
**Project:** Retail Shop Analysis Using Data Analytics

This notebook runs the queries in `sql/retail_analysis.sql` against the SQLite database
`dataset/retail.db` and explains each result. Build the database first:
```
python scripts/build_database.py
```
**Star schema used**
```
             dim_customer            dim_product
                   \                    /
                    \                  /
                     ----- fact_sales -----  dim_location
```

In [1]:
import re
import sqlite3
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_colwidth", 60)
con = sqlite3.connect("../dataset/retail.db")

# Read the named queries from the .sql file (blocks start with "-- @query name | description")
sql_text = open("../sql/retail_analysis.sql", encoding="utf-8").read()
queries = {}
for block in re.split(r"^-- @query ", sql_text, flags=re.M)[1:]:
    header, _, body = block.partition("\n")
    name, _, desc = header.partition("|")
    queries[name.strip()] = (desc.strip(), body.strip())
print(len(queries), "queries loaded:", ", ".join(queries))

def run(name, explanation):
    """Show the query, run it, display the result and the explanation."""
    desc, sql = queries[name]
    display(Markdown(f"### {desc}\n```sql\n{sql}\n```\n**What it does:** {explanation}"))
    result = pd.read_sql_query(sql, con)
    display(result)
    return result

17 queries loaded: 00_row_counts, 01_total_sales_profit, 02_top10_products_revenue, 03_top10_customers, 04_monthly_sales_trend, 05_yearly_growth, 06_category_performance, 07_region_sales, 08_low_sales_products, 09_average_order_value, 10_repeat_customers, 11_profit_margin_by_category, 12_loss_making_subcategories, 13_discount_impact, 14_ship_mode_performance, 15_top_product_per_category, 16_rfm_base


In [2]:
_ = run("00_row_counts", "Confirms that the load into the star schema worked: 8,399 fact rows, one row per customer, product and province.")

### Data-load check: row counts in each table
```sql
SELECT 'fact_sales'   AS table_name, COUNT(*) AS row_count FROM fact_sales   UNION ALL
SELECT 'dim_customer',               COUNT(*)              FROM dim_customer UNION ALL
SELECT 'dim_product',                COUNT(*)              FROM dim_product  UNION ALL
SELECT 'dim_location',               COUNT(*)              FROM dim_location;
```
**What it does:** Confirms that the load into the star schema worked: 8,399 fact rows, one row per customer, product and province.

,table_name,row_count
0,fact_sales,8399
1,dim_customer,795
2,dim_product,1263
3,dim_location,13


In [3]:
_ = run("01_total_sales_profit", "Aggregates the whole fact table. `COUNT(DISTINCT order_id)` counts orders rather than line items, and AOV = revenue ÷ orders.")

### Q1. Overall business KPIs
```sql
SELECT ROUND(SUM(sales), 2)                                 AS total_revenue,
       ROUND(SUM(profit), 2)                                AS total_profit,
       ROUND(100.0 * SUM(profit) / SUM(sales), 2)           AS profit_margin_pct,
       COUNT(DISTINCT order_id)                             AS total_orders,
       SUM(quantity)                                        AS total_units,
       COUNT(DISTINCT customer_id)                          AS total_customers,
       ROUND(SUM(sales) / COUNT(DISTINCT order_id), 2)      AS avg_order_value
FROM fact_sales;
```
**What it does:** Aggregates the whole fact table. `COUNT(DISTINCT order_id)` counts orders rather than line items, and AOV = revenue ÷ orders.

,total_revenue,total_profit,profit_margin_pct,total_orders,total_units,total_customers,avg_order_value
0,"14,915,600.82","1,521,767.98",10.20,5496,214777,795,"2,713.90"


In [4]:
_ = run("02_top10_products_revenue", "Joins facts to products, groups by product, sorts by revenue and keeps the top 10 with `LIMIT`.")

### Q2. Top 10 products by revenue
```sql
SELECT p.product_name,
       p.category,
       ROUND(SUM(f.sales), 2)                        AS revenue,
       ROUND(SUM(f.profit), 2)                       AS profit,
       SUM(f.quantity)                               AS units_sold,
       ROUND(100.0 * SUM(f.profit) / SUM(f.sales), 2) AS margin_pct
FROM fact_sales f
JOIN dim_product p ON p.product_id = f.product_id
GROUP BY p.product_id
ORDER BY revenue DESC
LIMIT 10;
```
**What it does:** Joins facts to products, groups by product, sorts by revenue and keeps the top 10 with `LIMIT`.

,product_name,category,revenue,profit,units_sold,margin_pct
0,Global Troy™ Executive Leather Low-Back Tilter,Furniture,"275,941.52","79,284.78",550,28.73
1,Polycom ViewStation™ ISDN Videoconferencing Unit,Technology,"255,303.97","-17,771.27",41,-6.96
2,Canon PC940 Copier,Technology,"210,910.68","47,705.85",474,22.62
3,"Riverside Palais Royal Lawyers Bookcase, Royale Cherry F...",Furniture,"206,494.13","14,291.57",236,6.92
4,Hewlett Packard LaserJet 3310 Copier,Technology,"194,880.35","73,299.61",334,37.61
5,Bretford CR8500 Series Meeting Room Furniture,Furniture,"147,412.21","-1,989.87",407,-1.35
6,Sharp AL-1530CS Digital Copier,Technology,"130,046.81","11,770.30",260,9.05
7,"Hewlett-Packard cp1700 [D, PS] Series Color Inkjet Printers",Technology,"128,100.43","55,281.39",269,43.15
8,GBC DocuBind 200 Manual Binding Machine,Office Supplies,"120,342.74","50,462.53",286,41.93
9,Canon imageCLASS 2200 Advanced Copier,Technology,"114,173.41","-31,885.68",32,-27.93


In [5]:
_ = run("03_top10_customers", "Same pattern on the customer dimension. `MAX(order_date)` shows how recently each top customer bought.")

### Q3. Top 10 customers by total spending
```sql
SELECT c.customer_name,
       COUNT(DISTINCT f.order_id)                    AS orders,
       ROUND(SUM(f.sales), 2)                        AS total_spent,
       ROUND(SUM(f.profit), 2)                       AS profit_generated,
       ROUND(SUM(f.sales) / COUNT(DISTINCT f.order_id), 2) AS avg_order_value,
       MAX(f.order_date)                             AS last_order_date
FROM fact_sales f
JOIN dim_customer c ON c.customer_id = f.customer_id
GROUP BY c.customer_id
ORDER BY total_spent DESC
LIMIT 10;
```
**What it does:** Same pattern on the customer dimension. `MAX(order_date)` shows how recently each top customer bought.

,customer_name,orders,total_spent,profit_generated,avg_order_value,last_order_date
0,Emily Phan,8,"117,124.44","34,005.44","14,640.55",2012-11-26
1,Deborah Brumfield,12,"97,433.14","31,121.22","8,119.43",2012-01-03
2,Roy Skaria,18,"92,542.15","1,343.94","5,141.23",2012-12-04
3,Sylvia Foulston,18,"88,875.76","5,141.90","4,937.54",2012-10-12
4,Grant Carroll,17,"88,417.00","27,977.29","5,201.00",2012-11-11
5,Alejandro Grove,9,"83,561.93","20,588.92","9,284.66",2012-09-01
6,Darren Budd,26,"81,577.34","1,857.66","3,137.59",2012-11-14
7,Julia Barnett,12,"80,044.45","-1,956.09","6,670.37",2012-12-12
8,John Lucas,14,"79,696.19","12,725.08","5,692.58",2012-12-02
9,Liz MacKendrick,14,"76,306.43","20,397.24","5,450.46",2012-07-02


In [6]:
_ = run("04_monthly_sales_trend", "`strftime('%Y-%m', ...)` turns each date into a year-month key, giving 48 monthly points for the trend chart.")

### Q4. Monthly revenue, profit and orders
```sql
SELECT strftime('%Y-%m', order_date)        AS year_month,
       ROUND(SUM(sales), 2)                 AS revenue,
       ROUND(SUM(profit), 2)                AS profit,
       COUNT(DISTINCT order_id)             AS orders
FROM fact_sales
GROUP BY year_month
ORDER BY year_month;
```
**What it does:** `strftime('%Y-%m', ...)` turns each date into a year-month key, giving 48 monthly points for the trend chart.

,year_month,revenue,profit,orders
0,2009-01,"516,302.96","62,326.40",133
1,2009-02,"332,480.64","30,422.68",101
2,2009-03,"411,628.73","22,802.96",124
3,2009-04,"393,276.48","39,132.73",118
4,2009-05,"230,145.54","18,035.35",119
5,2009-06,"263,456.07","34,640.55",105
6,2009-07,"380,503.97","41,749.43",120
7,2009-08,"329,754.72","14,682.84",127
8,2009-09,"325,292.31","46,652.05",123
9,2009-10,"361,555.27","55,228.01",100


In [7]:
_ = run("05_yearly_growth", "A CTE computes yearly totals. The window function `LAG()` reads the previous year's revenue to compute growth %.")

### Q5. Year-over-year revenue growth (window function LAG)
```sql
WITH yearly AS (
    SELECT strftime('%Y', order_date) AS order_year,
           SUM(sales)  AS revenue,
           SUM(profit) AS profit
    FROM fact_sales
    GROUP BY order_year
)
SELECT order_year,
       ROUND(revenue, 2)                                            AS revenue,
       ROUND(profit, 2)                                             AS profit,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY order_year))
             / LAG(revenue) OVER (ORDER BY order_year), 2)          AS revenue_growth_pct
FROM yearly
ORDER BY order_year;
```
**What it does:** A CTE computes yearly totals. The window function `LAG()` reads the previous year's revenue to compute growth %.

,order_year,revenue,profit,revenue_growth_pct
0,2009,"4,209,139.46","434,538.73",NaN
1,2010,"3,549,680.80","363,871.48",-15.67
2,2011,"3,436,816.70","381,455.99",-3.18
3,2012,"3,719,963.86","341,901.78",8.24


In [8]:
_ = run("06_category_performance", "A scalar sub-query supplies total revenue, so each category's share can be calculated in the same query.")

### Q6. Category-wise performance and share of revenue
```sql
SELECT p.category,
       ROUND(SUM(f.sales), 2)                                       AS revenue,
       ROUND(SUM(f.profit), 2)                                      AS profit,
       SUM(f.quantity)                                              AS units_sold,
       ROUND(100.0 * SUM(f.sales) / (SELECT SUM(sales) FROM fact_sales), 2) AS revenue_share_pct,
       ROUND(100.0 * SUM(f.profit) / SUM(f.sales), 2)               AS margin_pct
FROM fact_sales f
JOIN dim_product p ON p.product_id = f.product_id
GROUP BY p.category
ORDER BY revenue DESC;
```
**What it does:** A scalar sub-query supplies total revenue, so each category's share can be calculated in the same query.

,category,revenue,profit,units_sold,revenue_share_pct,margin_pct
0,Technology,"5,984,248.18","886,313.52",52175,40.12,14.81
1,Furniture,"5,178,590.54","117,433.03",44324,34.72,2.27
2,Office Supplies,"3,752,762.10","518,021.43",118278,25.16,13.80


In [9]:
_ = run("07_region_sales", "`RANK() OVER (...)` ranks provinces by revenue and by margin. This shows provinces that sell a lot but earn little.")

### Q7. Region and province sales with ranking (window function RANK)
```sql
SELECT l.region,
       l.province,
       ROUND(SUM(f.sales), 2)                               AS revenue,
       ROUND(SUM(f.profit), 2)                              AS profit,
       ROUND(100.0 * SUM(f.profit) / SUM(f.sales), 2)       AS margin_pct,
       RANK() OVER (ORDER BY SUM(f.sales) DESC)             AS revenue_rank,
       RANK() OVER (ORDER BY SUM(f.profit) / SUM(f.sales) DESC) AS margin_rank
FROM fact_sales f
JOIN dim_location l ON l.location_id = f.location_id
GROUP BY l.location_id
ORDER BY revenue DESC;
```
**What it does:** `RANK() OVER (...)` ranks provinces by revenue and by margin. This shows provinces that sell a lot but earn little.

,region,province,revenue,profit,margin_pct,revenue_rank,margin_rank
0,Ontario,Ontario,"3,063,212.48","346,868.54",11.32,1,4
1,West,British Columbia,"1,892,757.78","145,062.13",7.66,2,9
2,West,Alberta,"1,704,791.49","151,946.48",8.91,3,8
3,Quebec,Quebec,"1,510,195.08","140,426.65",9.30,4,7
4,Prairie,Saskatchewan,"1,464,455.82","184,732.96",12.61,5,2
5,Prairie,Manitoba,"1,372,848.78","136,427.16",9.94,6,6
6,Yukon,Yukon,"975,867.37","73,849.21",7.57,7,11
7,Atlantic,Nova Scotia,"817,729.36","85,361.87",10.44,8,5
8,Northwest Territories,Northwest Territories,"800,847.33","100,653.08",12.57,9,3
9,Atlantic,New Brunswick,"684,211.52","115,351.94",16.86,10,1


In [10]:
_ = run("08_low_sales_products", "Sorting ascending finds the weakest products: candidates for delisting or clearance.")

### Q8. Low-performing products: bottom 15 by revenue (with units and orders)
```sql
SELECT p.product_name,
       p.category,
       p.sub_category,
       COUNT(DISTINCT f.order_id)    AS orders,
       SUM(f.quantity)               AS units_sold,
       ROUND(SUM(f.sales), 2)        AS revenue,
       ROUND(SUM(f.profit), 2)       AS profit
FROM fact_sales f
JOIN dim_product p ON p.product_id = f.product_id
GROUP BY p.product_id
ORDER BY revenue ASC
LIMIT 15;
```
**What it does:** Sorting ascending finds the weakest products: candidates for delisting or clearance.

,product_name,category,sub_category,orders,units_sold,revenue,profit
0,Alliance Rubber Bands,Office Supplies,Rubber Bands,1,6,14.85,-2.79
1,"Sony IBM Color Diskettes, 25/Pack",Technology,Computer Peripherals,2,26,31.29,-65.33
2,*Staples* Packaging Labels,Office Supplies,Labels,2,12,34.94,4.32
3,Avery 482,Office Supplies,Labels,1,16,44.46,5.27
4,Blackstonian Pencils,Office Supplies,Pens & Art Supplies,1,19,50.07,-0.53
5,Colorific® Watercolor Pencils,Office Supplies,Pens & Art Supplies,1,10,50.97,4.45
6,Avery 516,Office Supplies,Labels,1,10,70.02,22.62
7,Computer Printout Index Tabs,Office Supplies,Binders and Binder Accessories,1,46,75.60,-225.25
8,Newell 332,Office Supplies,Pens & Art Supplies,1,26,78.03,0.51
9,Rogers Handheld Barrel Pencil Sharpener,Office Supplies,Pens & Art Supplies,3,30,93.44,-75.67


In [11]:
_ = run("09_average_order_value", "A CTE first totals each order, then AVG gives the true average order value overall (UNION ALL) and per segment.")

### Q9. Average order value overall and by customer segment
```sql
WITH order_totals AS (
    SELECT order_id, segment, SUM(sales) AS order_value
    FROM fact_sales
    GROUP BY order_id, segment
)
SELECT 'All segments'                AS segment,
       COUNT(*)                      AS orders,
       ROUND(AVG(order_value), 2)    AS avg_order_value
FROM (SELECT order_id, SUM(sales) AS order_value FROM fact_sales GROUP BY order_id)
UNION ALL
SELECT segment, COUNT(*), ROUND(AVG(order_value), 2)
FROM order_totals
GROUP BY segment;
```
**What it does:** A CTE first totals each order, then AVG gives the true average order value overall (UNION ALL) and per segment.

,segment,orders,avg_order_value
0,All segments,5496,"2,713.90"
1,Consumer,1076,"2,847.22"
2,Corporate,2002,"2,746.71"
3,Home Office,1341,"2,658.29"
4,Small Business,1077,"2,588.97"


In [12]:
_ = run("10_repeat_customers", "Counts orders per customer and groups them into buckets with `CASE`. Almost all customers are repeat buyers.")

### Q10. Repeat vs one-time customers
```sql
WITH customer_orders AS (
    SELECT customer_id, COUNT(DISTINCT order_id) AS n_orders
    FROM fact_sales
    GROUP BY customer_id
)
SELECT CASE WHEN n_orders = 1              THEN '1 order (one-time)'
            WHEN n_orders BETWEEN 2 AND 4  THEN '2-4 orders'
            WHEN n_orders BETWEEN 5 AND 9  THEN '5-9 orders'
            ELSE '10+ orders' END                                  AS customer_type,
       COUNT(*)                                                    AS customers,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_orders), 2) AS pct_of_customers
FROM customer_orders
GROUP BY customer_type
ORDER BY MIN(n_orders);
```
**What it does:** Counts orders per customer and groups them into buckets with `CASE`. Almost all customers are repeat buyers.

,customer_type,customers,pct_of_customers
0,1 order (one-time),11,1.38
1,2-4 orders,215,27.04
2,5-9 orders,410,51.57
3,10+ orders,159,20.00


In [13]:
_ = run("11_profit_margin_by_category", "Margin = profit ÷ sales × 100, at sub-category level within each category.")

### Q11. Profit margin by category and sub-category
```sql
SELECT p.category,
       p.sub_category,
       ROUND(SUM(f.sales), 2)                           AS revenue,
       ROUND(SUM(f.profit), 2)                          AS profit,
       ROUND(100.0 * SUM(f.profit) / SUM(f.sales), 2)   AS margin_pct
FROM fact_sales f
JOIN dim_product p ON p.product_id = f.product_id
GROUP BY p.category, p.sub_category
ORDER BY p.category, margin_pct DESC;
```
**What it does:** Margin = profit ÷ sales × 100, at sub-category level within each category.

,category,sub_category,revenue,profit,margin_pct
0,Furniture,Office Furnishings,"698,093.81","100,427.93",14.39
1,Furniture,Chairs & Chairmats,"1,761,836.55","149,649.73",8.49
2,Furniture,Bookcases,"822,652.04","-33,582.13",-4.08
3,Furniture,Tables,"1,896,008.14","-99,062.50",-5.22
4,Office Supplies,Labels,"38,981.55","13,677.17",35.09
5,Office Supplies,Binders and Binder Accessories,"1,022,957.59","307,413.39",30.05
6,Office Supplies,Envelopes,"174,085.80","48,182.60",27.68
7,Office Supplies,Appliances,"736,991.54","97,158.06",13.18
8,Office Supplies,Paper,"446,452.86","45,263.20",10.14
9,Office Supplies,Pens & Art Supplies,"167,107.22","7,564.78",4.53


In [14]:
_ = run("12_loss_making_subcategories", "`HAVING` filters groups after aggregation. Only sub-categories with a negative total profit remain.")

### Q12. Sub-categories with a net loss (HAVING clause)
```sql
SELECT category, sub_category,
       ROUND(SUM(sales), 2)  AS revenue,
       ROUND(SUM(profit), 2) AS profit
FROM v_sales
GROUP BY category, sub_category
HAVING SUM(profit) < 0
ORDER BY profit;
```
**What it does:** `HAVING` filters groups after aggregation. Only sub-categories with a negative total profit remain.

,category,sub_category,revenue,profit
0,Furniture,Tables,"1,896,008.14","-99,062.50"
1,Furniture,Bookcases,"822,652.04","-33,582.13"
2,Office Supplies,"Scissors, Rulers and Trimmers","80,996.31","-7,799.25"
3,Office Supplies,Rubber Bands,"15,006.63",-102.67


In [15]:
_ = run("13_discount_impact", "`CASE` bands the discount. Margin falls and the share of loss-making lines rises as discount increases.")

### Q13. Impact of discount level on profitability (CASE banding)
```sql
SELECT CASE WHEN discount <= 0.02 THEN '1) 0-2%'
            WHEN discount <= 0.05 THEN '2) 3-5%'
            WHEN discount <= 0.08 THEN '3) 6-8%'
            ELSE '4) 9%+' END                                          AS discount_band,
       COUNT(*)                                                        AS line_items,
       ROUND(SUM(sales), 2)                                            AS revenue,
       ROUND(100.0 * SUM(profit) / SUM(sales), 2)                      AS margin_pct,
       ROUND(100.0 * SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS loss_line_pct
FROM fact_sales
GROUP BY discount_band
ORDER BY discount_band;
```
**What it does:** `CASE` bands the discount. Margin falls and the share of loss-making lines rises as discount increases.

,discount_band,line_items,revenue,margin_pct,loss_line_pct
0,1) 0-2%,2327,"4,344,048.33",12.36,47.19
1,2) 3-5%,2335,"4,082,476.70",11.74,48.78
2,3) 6-8%,2209,"3,983,044.76",7.97,52.97
3,4) 9%+,1528,"2,506,031.03",7.50,56.09


In [16]:
_ = run("14_ship_mode_performance", "`julianday()` gives date differences in days. Delivery Truck carries the most revenue but has the lowest margin.")

### Q14. Shipping mode: volume, cost and margin
```sql
SELECT ship_mode,
       COUNT(*)                                          AS line_items,
       ROUND(SUM(sales), 2)                              AS revenue,
       ROUND(SUM(shipping_cost), 2)                      AS shipping_cost,
       ROUND(100.0 * SUM(profit) / SUM(sales), 2)        AS margin_pct,
       ROUND(AVG(julianday(ship_date) - julianday(order_date)), 2) AS avg_ship_days
FROM fact_sales
GROUP BY ship_mode
ORDER BY revenue DESC;
```
**What it does:** `julianday()` gives date differences in days. Delivery Truck carries the most revenue but has the lowest margin.

,ship_mode,line_items,revenue,shipping_cost,margin_pct,avg_ship_days
0,Regular Air,6270,"7,506,303.21","48,008.19",14.72,2.04
1,Delivery Truck,1146,"6,224,878.72","51,971.94",4.33,2.04
2,Express Air,983,"1,184,418.90","7,850.91",12.45,2.00


In [17]:
_ = run("15_top_product_per_category", "`ROW_NUMBER() OVER (PARTITION BY category ...)` numbers products within each category. Keeping rn = 1 gives each category's best seller.")

### Q15. Best-selling product in each category (ROW_NUMBER)
```sql
WITH ranked AS (
    SELECT category, product_name, SUM(sales) AS revenue,
           ROW_NUMBER() OVER (PARTITION BY category ORDER BY SUM(sales) DESC) AS rn
    FROM v_sales
    GROUP BY category, product_name
)
SELECT category, product_name, ROUND(revenue, 2) AS revenue
FROM ranked
WHERE rn = 1
ORDER BY revenue DESC;
```
**What it does:** `ROW_NUMBER() OVER (PARTITION BY category ...)` numbers products within each category. Keeping rn = 1 gives each category's best seller.

,category,product_name,revenue
0,Furniture,Global Troy™ Executive Leather Low-Back Tilter,"275,941.52"
1,Technology,Polycom ViewStation™ ISDN Videoconferencing Unit,"255,303.97"
2,Office Supplies,GBC DocuBind 200 Manual Binding Machine,"120,342.74"


In [18]:
rfm_sql = run("16_rfm_base", "Computes Recency, Frequency and Monetary per customer in SQL. This is the same input used by notebook 03.")
rfm_py = pd.read_csv("../results/rfm_segments.csv").set_index("customer_name")
check = rfm_sql.set_index("customer_name").join(rfm_py[["recency", "frequency", "monetary"]], rsuffix="_py")
print("SQL vs Python RFM values identical:",
      bool((check["recency_days"] == check["recency"]).all()
           and (check["frequency"] == check["frequency_py"]).all()
           and ((check["monetary"] - check["monetary_py"]).abs() < 0.01).all()))

### Q16. RFM base table per customer (input to customer segmentation)
```sql
SELECT c.customer_name,
       CAST(julianday((SELECT MAX(order_date) FROM fact_sales)) - julianday(MAX(f.order_date)) + 1
            AS INTEGER)                    AS recency_days,
       COUNT(DISTINCT f.order_id)          AS frequency,
       ROUND(SUM(f.sales), 2)              AS monetary
FROM fact_sales f
JOIN dim_customer c ON c.customer_id = f.customer_id
GROUP BY c.customer_id
ORDER BY monetary DESC;
```
**What it does:** Computes Recency, Frequency and Monetary per customer in SQL. This is the same input used by notebook 03.

,customer_name,recency_days,frequency,monetary
0,Emily Phan,35,8,"117,124.44"
1,Deborah Brumfield,363,12,"97,433.14"
2,Roy Skaria,27,18,"92,542.15"
3,Sylvia Foulston,80,18,"88,875.76"
4,Grant Carroll,50,17,"88,417.00"
...,...,...,...,...
790,Dorothy Dickinson,969,1,198.08
791,Katrina Edelman,574,2,180.76
792,Nicole Fjeld,852,2,153.03
793,Natalie DeCherney,931,1,125.90


SQL vs Python RFM values identical: True


C:\Users\asus\AppData\Local\Temp\ipykernel_12760\2072361438.py:3: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  check = rfm_sql.set_index("customer_name").join(rfm_py[["recency", "frequency", "monetary"]], rsuffix="_py")


In [19]:
con.close()